# ⭐ Phase 5 — Scoring de Réputation par Entité

**Projet : African Media Intelligence AI**  
**Auteur : Esmel Amari (Phil)**  
**Description :** Ce notebook calcule un **indicateur de réputation composite** pour chaque entité (entreprise, gouvernement, organisation) identifiée dans les articles. Le score combine fréquence de mention, sentiment associé et récence.

---

### Formule du RepScore

```
RepScore = ((%Positif × 1) + (%Neutre × 0) + (%Négatif × -1))
           × log(1 + Fréquence)
           × Facteur_Récence
```

- **Intervalle** : -100 à +100
- **> +30** : Réputation forte
- **-10 à +30** : Réputation neutre
- **< -10** : Réputation négative (alerte)


## 0. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import json
import warnings
import logging
from pathlib import Path
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

BASE_DIR = Path('..').resolve()
PROC_DIR = BASE_DIR / 'data' / 'processed'

# ── Paramètres ────────────────────────────────────────────────────
MIN_MENTIONS      = 2     # Mentions minimales pour inclure une entité
RECENCY_DAYS      = 30    # Fenêtre de récence (jours)
RECENCY_WEIGHT    = 0.3   # Poids du facteur récence dans le score final

print("✅ Configuration chargée")

✅ Configuration chargée


## 1. Chargement des Données

In [2]:
# ── Table entités long-format (Phase 4) ───────────────────────────
entity_files = sorted(PROC_DIR.glob('entities_flat_*.csv'), reverse=True)
if not entity_files:
    raise FileNotFoundError("Fichier entities_flat_*.csv introuvable. Exécutez 04_ner_entities.ipynb")

df_ent = pd.read_csv(entity_files[0], encoding='utf-8-sig')
print(f"✅ Entités chargées : {entity_files[0].name}")
print(f"   → {len(df_ent)} lignes | {df_ent['entite'].nunique()} entités uniques")

# ── Articles principaux (Phase 4) ─────────────────────────────────
article_files = sorted(PROC_DIR.glob('articles_ner_*.csv'), reverse=True)
if not article_files:
    article_files = sorted(PROC_DIR.glob('articles_topics_*.csv'), reverse=True)

if article_files:
    df_articles = pd.read_csv(article_files[0], encoding='utf-8-sig')
    print(f"✅ Articles chargés : {article_files[0].name} → {len(df_articles)} articles")
else:
    df_articles = None
    print("⚠️  Pas de fichier articles trouvé (RepScore sera calculé depuis entities_flat)")

✅ Entités chargées : entities_flat_20260512_2037.csv
   → 532 lignes | 314 entités uniques
✅ Articles chargés : articles_ner_20260512_2037.csv → 121 articles


## 2. Nettoyage & Standardisation

In [3]:
# ── Normalisation du sentiment ─────────────────────────────────────
SENTIMENT_MAP = {
    'positif'  : 'Positif',
    'positive' : 'Positif',
    'neutre'   : 'Neutre',
    'neutral'  : 'Neutre',
    'négatif'  : 'Négatif',
    'negative' : 'Négatif',
    'negatif'  : 'Négatif',
}
df_ent['sentiment'] = (
    df_ent['sentiment']
    .fillna('Neutre')
    .str.lower()
    .map(SENTIMENT_MAP)
    .fillna('Neutre')
)

# ── Nettoyage des entités ──────────────────────────────────────────
df_ent['entite'] = df_ent['entite'].str.strip()
df_ent = df_ent[df_ent['entite'].str.len() >= 3].copy()

# ── Parsing de la date ─────────────────────────────────────────────
df_ent['date_publication'] = pd.to_datetime(
    df_ent['date_publication'], errors='coerce'
)
df_ent['date_pub_str'] = df_ent['date_publication'].dt.strftime('%Y-%m-%d').fillna('N/A')

print(f"✅ Données nettoyées : {len(df_ent)} mentions valides")
print(f"\nSentiments présents :")
print(df_ent['sentiment'].value_counts().to_string())

✅ Données nettoyées : 511 mentions valides

Sentiments présents :
sentiment
Neutre    511


## 3. Calcul du RepScore

In [4]:
def compute_recency_factor(dates_series: pd.Series, reference_date=None, window_days=30) -> float:
    """
    Calcule le facteur de récence (0 à 1).
    1.0 = toutes les mentions récentes | 0.0 = toutes très anciennes
    """
    if reference_date is None:
        reference_date = datetime.now()
    valid = dates_series.dropna()
    if len(valid) == 0:
        return 0.5
    # Proportion de mentions dans la fenêtre de récence
    cutoff = reference_date - timedelta(days=window_days)
    recent = (valid >= pd.Timestamp(cutoff)).sum()
    return round(recent / len(valid), 4)


def compute_rep_score(pos_pct, neg_pct, frequency, recency_factor) -> float:
    """
    Score de réputation composite [-100, +100].
    """
    sentiment_score = (pos_pct - neg_pct)  # -1 à +1
    freq_weight     = np.log1p(frequency)   # Atténuation log
    recency_adj     = 1 + RECENCY_WEIGHT * recency_factor  # 1 à 1.3
    score = sentiment_score * freq_weight * recency_adj * 10
    return round(np.clip(score, -100, 100), 2)


def classify_reputation(score: float) -> str:
    if score >= 30:
        return '🟢 Positive'
    elif score >= -10:
        return '🟡 Neutre'
    else:
        return '🔴 Négative'


print("✅ Fonctions de scoring définies")

✅ Fonctions de scoring définies


## 4. Agrégation par Entité

In [5]:
rep_records = []
ref_date = datetime.now()

grouped = df_ent.groupby(['entite', 'type_entite'])

for (entite, type_ent), group in grouped:
    freq = len(group)
    if freq < MIN_MENTIONS:
        continue

    # Distribution sentiment
    sent_counts = group['sentiment'].value_counts()
    total = freq

    pos_count = sent_counts.get('Positif', 0)
    neu_count = sent_counts.get('Neutre',  0)
    neg_count = sent_counts.get('Négatif', 0)

    pos_pct = pos_count / total
    neu_pct = neu_count / total
    neg_pct = neg_count / total

    # Récence
    recency = compute_recency_factor(group['date_publication'], ref_date, RECENCY_DAYS)

    # RepScore
    rep_score = compute_rep_score(pos_pct, neg_pct, freq, recency)

    # Sources citantes
    sources = ', '.join(sorted(group['source'].dropna().unique()))

    # Topics associés
    topics = ', '.join(
        group['topic_label'].dropna().value_counts().head(2).index.tolist()
    ) if 'topic_label' in group.columns else ''

    # Dernière mention
    last_mention = group['date_pub_str'].max()

    rep_records.append({
        'entite'           : entite,
        'type_entite'      : type_ent,
        'nb_mentions'      : freq,
        'nb_positif'       : int(pos_count),
        'nb_neutre'        : int(neu_count),
        'nb_negatif'       : int(neg_count),
        'pct_positif'      : round(pos_pct * 100, 1),
        'pct_neutre'       : round(neu_pct * 100, 1),
        'pct_negatif'      : round(neg_pct * 100, 1),
        'facteur_recence'  : recency,
        'rep_score'        : rep_score,
        'statut_reputation': classify_reputation(rep_score),
        'sources_citantes' : sources,
        'topics_associes'  : topics,
        'derniere_mention' : last_mention
    })

df_rep = pd.DataFrame(rep_records).sort_values('rep_score', ascending=False)

print(f"✅ {len(df_rep)} entités scorées")
print(f"\nDistribution des statuts :")
print(df_rep['statut_reputation'].value_counts().to_string())

✅ 58 entités scorées

Distribution des statuts :
statut_reputation
🟡 Neutre    58


## 5. Analyse des Résultats

In [6]:
cols_display = ['entite','type_entite','nb_mentions','pct_positif','pct_negatif','rep_score','statut_reputation']

print("=" * 70)
print("TOP 10 — MEILLEURE RÉPUTATION")
print("=" * 70)
print(df_rep.head(10)[cols_display].to_string(index=False))

print("\n" + "=" * 70)
print("TOP 10 — RÉPUTATION LA PLUS NÉGATIVE (alertes)")
print("=" * 70)
print(df_rep.tail(10)[cols_display].to_string(index=False))

print("\n" + "=" * 70)
print("PAR TYPE D'ENTITÉ")
print("=" * 70)
summary = df_rep.groupby('type_entite').agg(
    nb_entites   = ('entite', 'count'),
    score_moyen  = ('rep_score', 'mean'),
    total_mentions = ('nb_mentions', 'sum')
).round(2)
print(summary.to_string())

TOP 10 — MEILLEURE RÉPUTATION
         entite type_entite  nb_mentions  pct_positif  pct_negatif  rep_score statut_reputation
        Abidjan         GPE            2          0.0          0.0        0.0          🟡 Neutre
            RDC         LOC            3          0.0          0.0        0.0          🟡 Neutre
           Mali         PER            3          0.0          0.0        0.0          🟡 Neutre
          Maroc         GPE            3          0.0          0.0        0.0          🟡 Neutre
          Monde         ORG            4          0.0          0.0        0.0          🟡 Neutre
        Nairobi         GPE           18          0.0          0.0        0.0          🟡 Neutre
        Nairobi         LOC           11          0.0          0.0        0.0          🟡 Neutre
        Namibia         GPE            2          0.0          0.0        0.0          🟡 Neutre
Nicolas Sarkozy         PER            2          0.0          0.0        0.0          🟡 Neutre
          

## 6. Évolution Temporelle du Sentiment

In [7]:
# ── Tendance sentiment par jour ────────────────────────────────────
df_temp = df_ent.copy()
df_temp['date'] = pd.to_datetime(df_temp['date_publication'], errors='coerce').dt.date
df_temp = df_temp.dropna(subset=['date'])

evolution = (
    df_temp.groupby(['date', 'sentiment'])
    .size()
    .reset_index(name='count')
    .pivot(index='date', columns='sentiment', values='count')
    .fillna(0)
    .reset_index()
)

# Assurer l'existence des 3 colonnes
for col in ['Positif', 'Neutre', 'Négatif']:
    if col not in evolution.columns:
        evolution[col] = 0

evolution.columns.name = None
evolution = evolution.sort_values('date')

print("=== ÉVOLUTION SENTIMENT (5 derniers jours) ===")
print(evolution.tail(5).to_string(index=False))

print(f"\n✅ {len(evolution)} jours d'évolution calculés")

=== ÉVOLUTION SENTIMENT (5 derniers jours) ===
      date  Neutre  Positif  Négatif
2026-05-09       9        0        0
2026-05-10       9        0        0
2026-05-11      64        0        0
2026-05-12     147        0        0
2026-05-13      55        0        0

✅ 23 jours d'évolution calculés


## 7. Sauvegarde — Données pour Export Power BI

In [8]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M')

# ── 1. Scores de réputation ────────────────────────────────────────
rep_file = PROC_DIR / f'reputation_scores_{timestamp}.csv'
df_rep.to_csv(rep_file, index=False, encoding='utf-8-sig')
print(f"✅ Reputation scores : {rep_file}")

# ── 2. Évolution temporelle ────────────────────────────────────────
evo_file = PROC_DIR / f'sentiment_evolution_{timestamp}.csv'
evolution.to_csv(evo_file, index=False, encoding='utf-8-sig')
print(f"✅ Évolution sentiment : {evo_file}")

# ── 3. Metadata du projet ──────────────────────────────────────────
meta = {
    'projet'            : 'African Media Intelligence AI',
    'auteur'            : 'Esmel Amari (Phil)',
    'date_generation'   : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_articles'    : int(len(df_articles)) if df_articles is not None else 'N/A',
    'total_entites'     : int(len(df_rep)),
    'total_mentions'    : int(len(df_ent)),
    'periode_analyse'   : f"{df_ent['date_pub_str'].min()} → {df_ent['date_pub_str'].max()}",
    'fichiers_generes'  : [
        str(rep_file.name),
        str(evo_file.name),
        str(entity_files[0].name)
    ]
}
meta_file = PROC_DIR / 'project_metadata.json'
with open(meta_file, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print(f"✅ Métadonnées : {meta_file}")

print(f"\n🎯 Phase 5 complète — Prêt pour l'export Excel Power BI")
print(f"   → Exécutez maintenant : exports/powerbi_export.py")

✅ Reputation scores : C:\Users\E682\Desktop\data\processed\reputation_scores_20260512_2039.csv
✅ Évolution sentiment : C:\Users\E682\Desktop\data\processed\sentiment_evolution_20260512_2039.csv
✅ Métadonnées : C:\Users\E682\Desktop\data\processed\project_metadata.json

🎯 Phase 5 complète — Prêt pour l'export Excel Power BI
   → Exécutez maintenant : exports/powerbi_export.py


---

## ✅ Phase 5 Terminée

**Ce qui a été accompli :**
- ✅ RepScore composite calculé pour chaque entité
- ✅ Classement Positive / Neutre / Négative
- ✅ Évolution temporelle du sentiment générée
- ✅ Tous les fichiers CSV prêts pour Power BI

**Prochaine étape :** `exports/powerbi_export.py` — Consolidation en fichier Excel multi-onglets pour Power BI
